# How to Use This Notebook

This notebook is designed to be run sequentially from top to bottom without manual intervention. The cells are grouped into numbered steps. Please execute each step in order to run the verification.

- **Step 1: Environment Setup:** Prepares the Kaggle environment, installs dependencies, and verifies TPU access.
- **Step 2: Apply Compatibility Fix:** Downgrades NumPy to prevent known version conflicts.
- **Step 3: Configure Checkpoint Path:** Finds the Llama 3.1 checkpoint dataset and sets the required environment variable.
- **Step 4: Generate Config and Run Verification:** Uses the verified `load_parameters_path` key to generate the complete YAML file and runs the final 1-step training verification. Success is indicated by a "Verification run completed successfully" message.


# Step 1: Environment Setup

This step prepares the Kaggle environment by:
1.  **Verifying JAX and TPU Access:** Ensures the notebook can see the 8 TPU devices.
2.  **Cloning MaxText:** Clones the `google/maxtext` repository, which contains the training scripts.
3.  **Installing Dependencies:** Installs all Python packages required by MaxText from `requirements.txt`.


In [1]:
import os, sys, platform, subprocess

print("Verifying JAX and TPU environment...")
try:
    import jax
    import jax.numpy as jnp
    device_count = jax.device_count()
    print(f"✅ JAX version: {jax.__version__}")
    print(f"✅ Detected {device_count} TPU devices.")
    if device_count != 8:
        print("⚠️ WARNING: Expected 8 TPU devices, but found a different number.")
except Exception as e:
    print(f"❌ ERROR: JAX/TPU verification failed: {e}")
    raise

print("\nCloning MaxText repository...")
if os.path.exists('maxtext'):
    print("✅ 'maxtext' already exists. Skipping clone.")
else:
    subprocess.run(["git", "clone", "https://github.com/google/maxtext.git"], check=True)
    print("✅ MaxText repository cloned.")

# Pin to MaxText commit compatible with JAX 0.4.34 (NVIDIA JAX Release 25.01)
stable_commit_hash = "4651cb3c73de"
print(f"\nChecking out MaxText commit compatible with JAX 0.4.34: {stable_commit_hash}")
try:
    subprocess.run(["git", "checkout", stable_commit_hash], check=True, cwd="maxtext")
    print("✅ Git checkout successful.")
except subprocess.CalledProcessError as e:
    print(f"❌ Git checkout failed: {e}")
    raise

print("\nInstalling dependencies...")
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "pkg-config"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "maxtext/requirements.txt"], check=True, capture_output=True)
print("✅ Dependencies installed.")


Verifying JAX and TPU environment...


E0000 00:00:1759412955.737161      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:230


✅ JAX version: 0.4.34
✅ Detected 8 TPU devices.

Cloning MaxText repository...


Cloning into 'maxtext'...


✅ MaxText repository cloned.

Checking out MaxText commit compatible with JAX 0.4.34: 4651cb3c73de


Note: switching to '4651cb3c73de'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 4651cb3c Merge pull request #1099 from AI-Hypercomputer:mattdavidow-jdi-telemetry


✅ Git checkout successful.

Installing dependencies...
✅ Dependencies installed.


In [2]:
!grep -r "raw_keys\\['" /kaggle/working/maxtext/MaxText/pyconfig.py

/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


    max_logging.log(f"Running Model: {raw_keys['model_name']}")
      ), f"The number of layers per stage ({raw_keys['num_layers_per_pipeline_stage']}) times the number of stages ({num_stages}) must divide the number of decoder layers ({raw_keys['num_decoder_layers']}) "
    ), f"The product of pipeline stages ({num_stages}), repeats ({raw_keys['num_pipeline_repeats']}), and layers per stage ({raw_keys['num_layers_per_pipeline_stage']}) must be equal to the number of layers ({raw_keys['num_decoder_layers']})"
    ), f"The number of microbatches ({raw_keys['num_pipeline_microbatches']}) must be divisible by the number of stages ({num_stages})"
    ), f"The batch size ({raw_keys['micro_batch_size_to_train_on']}) must be divisible by the number of microbatches ({raw_keys['num_pipeline_microbatches']})"
      ), f"Delayed activation forwarding requires at least 2 * num_stages microbatches, but {num_stages} stages are used with {raw_keys['num_pipeline_microbatches']} microbatches"
        f

# Step 2: Apply Compatibility Fix

This step downgrades NumPy to version 1.26.4. This is a mandatory step to prevent known compatibility issues between the pre-installed TensorFlow and NumPy 2.x in the Kaggle TPU environment.


In [3]:
import sys
import subprocess

print("Applying NumPy compatibility fix...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"], check=True, capture_output=True)

print("Verifying NumPy version...")
# We run this in a subprocess to ensure we get the version from the updated environment
result = subprocess.run([sys.executable, "-c", "import numpy as np; print(np.__version__)"], check=True, capture_output=True, text=True)
numpy_version = result.stdout.strip()
print(f"✅ NumPy version is now: {numpy_version}")

if numpy_version != "1.26.4":
    print("⚠️ WARNING: Expected NumPy 1.26.4, but a different version is installed. This may cause issues.")
else:
    print("✅ NumPy version successfully downgraded to 1.26.4.")

print("\nNOTE: A kernel restart may be required for the version change to fully propagate in all contexts.")


Applying NumPy compatibility fix...
Verifying NumPy version...
✅ NumPy version is now: 1.26.4
✅ NumPy version successfully downgraded to 1.26.4.

NOTE: A kernel restart may be required for the version change to fully propagate in all contexts.


# Step 3: Configure Checkpoint Path

This step dynamically locates the pre-converted Llama 3.1 MaxText checkpoint within the attached Kaggle Datasets and sets the `MAXTEXT_CHECKPOINT_DIR` environment variable. This makes the checkpoint path available for the final verification step.


In [4]:
import os
from pathlib import Path

dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"Inspecting dataset directory: {dataset_path}")
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

required_files = ["_CHECKPOINT_METADATA", "items"]
if all((dataset_path / f).exists() for f in required_files):
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
else:
    raise FileNotFoundError(f"Could not find required checkpoint files in {dataset_path}")

os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


Inspecting dataset directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Checkpoint found in root directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR=/kaggle/input/llama-3-1-8b-maxtext-checkpoint


# Step 4: Generate Config and Run Verification

This is the final, fully automated step. It performs the following actions:

1.  **Sets `PYTHONPATH`:** Ensures the MaxText library can be correctly imported.
2.  **Generates YAML:** Creates the `verification_minimal.yml` file using the verified `load_parameters_path` key and the checkpoint path from the previous step.
3.  **Runs Verification:** Executes the MaxText training script as a module (`MaxText.train`) for a single step. 

A successful run will print "✅ Verification run completed successfully." and is the evidence that the entire environment is correctly configured.


In [ ]:
import os
import sys
import runpy
from pathlib import Path

# Set environment variable to resolve TensorFlow protobuf conflict
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Define paths and add MaxText to the Python path for in-process import
python_path = Path.cwd() / "maxtext"
if str(python_path) not in sys.path:
    sys.path.insert(0, str(python_path))
print(f"✅ Added to sys.path: {python_path}")

# 2. Get checkpoint path and define the verified key
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run the previous step first.")

verified_checkpoint_key = "load_parameters_path"
print(f"✅ Using verified key '{verified_checkpoint_key}' for checkpoint loading.")

# 3. Generate the complete and correct YAML configuration
config_text = f"""
# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
enable_checkpointing: False
hardware: 'tpu'
jax_cache_dir: "/kaggle/working/jax_cache"
dataset_type: "synthetic"
expansion_factor_real_data: 1.0
log_period: 100
scan_layers: True
attention: 'autoselected'
{verified_checkpoint_key}: "{checkpoint_path}"
compile_topology: False

# JAX/Debug defaults to avoid KeyErrors
jax_debug_log_modules: []
jax_disable_jit: False
jax_enable_x64: False
jax_debug_nans: False
jax_profile_server: ""

# Required model parameters for llama3.1-8b
model_name: "llama3.1-8b"
global_parameter_scale: 1
base_emb_dim: 4096
base_num_query_heads: 32
base_num_kv_heads: 8
base_num_decoder_layers: 32
base_mlp_dim: 14336
head_dim: 128
vocab_size: 128256
max_target_length: 2048
max_prefill_predict_length: 1024
normalization_layer_epsilon: 0.00001
decoder_block: llama2
enable_dropout: False
logits_via_embedding: False
mlp_activations: ['silu', 'linear']
dtype: "bfloat16"
attn_logits_soft_cap: 0
final_logits_soft_cap: 0
rope_max_timescale: 500000

# Training configuration
learning_rate: 0.0001
learning_rate_schedule_steps: -1
warmup_steps_fraction: 0.01
cosine_learning_rate_final_fraction: 0.1
gradient_clipping_threshold: 1.0
adam_b1: 0.9
adam_b2: 0.95
adam_eps: 0.00000001
adam_weight_decay: 0.1
init_weights_seed: 0
"""

config_path = Path("/kaggle/working/verification_minimal.yml")
config_path.write_text(config_text)
print(f"✅ Wrote config to: {config_path}")
print("--- Config Contents ---")
print(config_text)
print("-----------------------")

# 4. Run the verification script IN-PROCESS using runpy
print("\\n🚀 Running 1-step verification (in-process)...")

# Temporarily replace sys.argv for the script
original_argv = sys.argv
try:
    # Set argv for the script to run. The first element is the script name,
    # followed by its arguments.
    sys.argv = ['MaxText/train.py', str(config_path)]
    
    # Run the module. 'run_name="__main__"' makes the script believe it's being
    # executed directly.
    runpy.run_module('MaxText.train', run_name='__main__')
    
    print("\\n✅ Verification run completed successfully.")
except SystemExit as e:
    if e.code == 0:
        print("\\n✅ Verification run completed successfully (SystemExit code 0).")
    else:
        # The script likely failed and called sys.exit()
        print(f"\\n❌ Verification run failed with SystemExit code: {e.code}")
except Exception as e:
    # Catch any other unexpected exceptions
    import traceback
    print(f"\\n❌ An unexpected error occurred: {e}")
    traceback.print_exc()
finally:
    # Always restore the original sys.argv
    sys.argv = original_argv

SyntaxError: unexpected character after line continuation character (3802344442.py, line 24)